In [1]:
import random

# ============================================================
# RNA-RNA Interaction Prediction using Genetic Algorithm
# ============================================================

RNA_BASES = "AUCG"

# Approximate pairing energies.
# More negative = more stable interaction.
PAIR_ENERGY = {
    ("A", "U"): -2.0,
    ("U", "A"): -2.0,
    ("G", "C"): -3.0,
    ("C", "G"): -3.0,
    ("G", "U"): -1.0,   # wobble pair
    ("U", "G"): -1.0
}


# ------------------------------------------------------------
# Validate RNA sequence
# ------------------------------------------------------------

def validate_rna(sequence):
    sequence = sequence.upper().strip()

    if not sequence:
        raise ValueError("RNA sequence cannot be empty.")

    if any(base not in RNA_BASES for base in sequence):
        raise ValueError(
            "RNA sequence can contain only A, U, C and G."
        )

    return sequence


# ------------------------------------------------------------
# Calculate interaction energy
# ------------------------------------------------------------

def interaction_energy(rna1, rna2):
    """
    Calculate approximate hybridization energy.

    rna2 is considered in antiparallel orientation.
    """

    rna2 = rna2[::-1]

    energy = 0.0
    paired_bases = 0

    for b1, b2 in zip(rna1, rna2):

        if (b1, b2) in PAIR_ENERGY:
            energy += PAIR_ENERGY[(b1, b2)]
            paired_bases += 1

    return energy, paired_bases


# ------------------------------------------------------------
# Chromosome representation
# ------------------------------------------------------------

class Chromosome:

    def __init__(self, start1, start2, length):
        self.start1 = start1
        self.start2 = start2
        self.length = length

        self.energy = None
        self.pairs = None
        self.fitness = None

    def evaluate(self, rna1, rna2):

        fragment1 = rna1[
            self.start1:self.start1 + self.length
        ]

        fragment2 = rna2[
            self.start2:self.start2 + self.length
        ]

        energy, pairs = interaction_energy(
            fragment1,
            fragment2
        )

        self.energy = energy
        self.pairs = pairs

        # Reward stable interactions.
        # More negative energy -> larger fitness.
        self.fitness = (
            pairs * 2.0
            - energy
        )

        return self.fitness


# ------------------------------------------------------------
# Create initial population
# ------------------------------------------------------------

def create_population(
    rna1,
    rna2,
    population_size,
    min_length,
    max_length
):

    population = []

    max_length = min(
        max_length,
        len(rna1),
        len(rna2)
    )

    for _ in range(population_size):

        length = random.randint(
            min_length,
            max_length
        )

        start1 = random.randint(
            0,
            len(rna1) - length
        )

        start2 = random.randint(
            0,
            len(rna2) - length
        )

        chromosome = Chromosome(
            start1,
            start2,
            length
        )

        chromosome.evaluate(rna1, rna2)

        population.append(chromosome)

    return population


# ------------------------------------------------------------
# Tournament selection
# ------------------------------------------------------------

def tournament_selection(
    population,
    tournament_size=3
):

    candidates = random.sample(
        population,
        tournament_size
    )

    return max(
        candidates,
        key=lambda x: x.fitness
    )


# ------------------------------------------------------------
# Crossover
# ------------------------------------------------------------

def crossover(parent1, parent2):

    child_start1 = random.choice([
        parent1.start1,
        parent2.start1
    ])

    child_start2 = random.choice([
        parent1.start2,
        parent2.start2
    ])

    child_length = random.choice([
        parent1.length,
        parent2.length
    ])

    return Chromosome(
        child_start1,
        child_start2,
        child_length
    )


# ------------------------------------------------------------
# Mutation
# ------------------------------------------------------------

def mutate(
    chromosome,
    rna1,
    rna2,
    mutation_rate=0.1
):

    if random.random() < mutation_rate:

        chromosome.start1 = random.randint(
            0,
            len(rna1) - chromosome.length
        )

    if random.random() < mutation_rate:

        chromosome.start2 = random.randint(
            0,
            len(rna2) - chromosome.length
        )

    if random.random() < mutation_rate:

        chromosome.length += random.choice([-1, 1])

        chromosome.length = max(
            3,
            chromosome.length
        )

        chromosome.length = min(
            chromosome.length,
            len(rna1) - chromosome.start1,
            len(rna2) - chromosome.start2
        )


# ------------------------------------------------------------
# Genetic Algorithm
# ------------------------------------------------------------

def genetic_algorithm(
    rna1,
    rna2,
    population_size=100,
    generations=200,
    min_length=5,
    max_length=20,
    mutation_rate=0.1,
    elite_size=5
):

    population = create_population(
        rna1,
        rna2,
        population_size,
        min_length,
        max_length
    )

    best_solution = None

    for generation in range(generations):

        # Evaluate population
        for chromosome in population:

            chromosome.evaluate(
                rna1,
                rna2
            )

        # Sort by fitness
        population.sort(
            key=lambda x: x.fitness,
            reverse=True
        )

        # Store best solution
        if (
            best_solution is None
            or population[0].fitness
            > best_solution.fitness
        ):

            best_solution = population[0]

        # Elitism
        new_population = population[
            :elite_size
        ]

        # Generate remaining population
        while len(new_population) < population_size:

            parent1 = tournament_selection(
                population
            )

            parent2 = tournament_selection(
                population
            )

            child = crossover(
                parent1,
                parent2
            )

            mutate(
                child,
                rna1,
                rna2,
                mutation_rate
            )

            child.evaluate(
                rna1,
                rna2
            )

            new_population.append(child)

        population = new_population

        # Progress
        if generation % 20 == 0:

            print(
                f"Generation {generation:3d} | "
                f"Best fitness = "
                f"{best_solution.fitness:.2f} | "
                f"Energy = "
                f"{best_solution.energy:.2f}"
            )

    return best_solution


# ------------------------------------------------------------
# Display result
# ------------------------------------------------------------

def display_result(
    best,
    rna1,
    rna2
):

    fragment1 = rna1[
        best.start1:
        best.start1 + best.length
    ]

    fragment2 = rna2[
        best.start2:
        best.start2 + best.length
    ]

    reverse_fragment2 = fragment2[::-1]

    print("\n================================")
    print("RNA-RNA INTERACTION PREDICTION")
    print("================================")

    print(
        f"RNA 1 position : "
        f"{best.start1} - "
        f"{best.start1 + best.length - 1}"
    )

    print(
        f"RNA 2 position : "
        f"{best.start2} - "
        f"{best.start2 + best.length - 1}"
    )

    print(
        f"\nRNA 1 : {fragment1}"
    )

    print(
        f"RNA 2 : {reverse_fragment2}"
    )

    print(
        f"\nBase pairs     : {best.pairs}"
    )

    print(
        f"Energy         : "
        f"{best.energy:.2f}"
    )

    print(
        f"Fitness        : "
        f"{best.fitness:.2f}"
    )


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":

    # Example RNA sequences
    rna1 = "GCGCUAUCGAAUGCGCUACG"
    rna2 = "CGUAGCGCAUUCGAGCGC"

    rna1 = validate_rna(rna1)
    rna2 = validate_rna(rna2)

    best = genetic_algorithm(
        rna1,
        rna2,
        population_size=100,
        generations=200,
        min_length=5,
        max_length=15,
        mutation_rate=0.10,
        elite_size=5
    )

    display_result(
        best,
        rna1,
        rna2
    )


Generation   0 | Best fitness = 64.00 | Energy = -36.00
Generation  20 | Best fitness = 64.00 | Energy = -36.00
Generation  40 | Best fitness = 64.00 | Energy = -36.00
Generation  60 | Best fitness = 64.00 | Energy = -36.00
Generation  80 | Best fitness = 64.00 | Energy = -36.00
Generation 100 | Best fitness = 64.00 | Energy = -36.00
Generation 120 | Best fitness = 64.00 | Energy = -36.00
Generation 140 | Best fitness = 64.00 | Energy = -36.00
Generation 160 | Best fitness = 64.00 | Energy = -36.00
Generation 180 | Best fitness = 64.00 | Energy = -36.00

RNA-RNA INTERACTION PREDICTION
RNA 1 position : 5 - 19
RNA 2 position : 0 - 14

RNA 1 : AUCGAAUGCGCUACG
RNA 2 : GAGCUUACGCGAUGC

Base pairs     : 14
Energy         : -36.00
Fitness        : 64.00
